# End-to-End Two-Stage HPO

**1 Trial = CLF → 집계 → FS → REG → RMSE**

- 단일 Optuna objective에서 전체 파이프라인 실행
- CLF/FS/REG 하이퍼파라미터를 동시에 최적화 (전역 최적)
- 웨이퍼맵 기반 피처 사전 필터링 적용

## 1. 환경 설정 및 데이터 로드

In [ ]:
# ============================================================
# 환경 설정 + 데이터 로드
# ============================================================
import os, sys

# --- Google Drive 파일 ID ---
E2E_GDRIVE_ID = None  # ← e2e.zip 업로드 후 ID 입력

try:
    import google.colab
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system('gdown --id 1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system('gdown --id 1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system('gdown --id 1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    # E2E 모듈 다운로드
    e2e_target = '/content/project/3_modeling/e2e_pipeline/modules/e2e_hpo.py'
    if not os.path.exists(e2e_target):
        assert E2E_GDRIVE_ID is not None, (
            'E2E_GDRIVE_ID를 설정하세요! '
            'e2e_pipeline/ 폴더를 zip으로 만들어 Google Drive에 업로드 후 ID 입력'
        )
        os.system(f'gdown --id {E2E_GDRIVE_ID} -O /content/e2e.zip')
        os.system('unzip -qo /content/e2e.zip -d /content/project/3_modeling')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../setup.py

# --- 기본 라이브러리 ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# --- 프로젝트 유틸 ---
from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, POSITION_COL
from utils.data import load_all, get_feat_cols, split_xs
from utils.evaluate import evaluate, postprocess
from utils.experiment import (
    log_experiment, check_exp_id, check_duplicate_params, download_from_drive,
)

# --- 전처리 모듈 ---
sys.path.insert(0, os.path.join(PROJECT_ROOT, '2_preprocessing'))
from cleaning import run_cleaning
from outlier import run_outlier_treatment

# --- E2E 모듈 ---
sys.path.insert(0, os.path.join(PROJECT_ROOT, '3_modeling', 'e2e_pipeline'))
from modules.e2e_hpo import run_e2e_optimization, rerun_best_trial

# --- 데이터 로드 ---
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)
ys_train = ys['train']

print(f'Feature 수: {len(feat_cols)}')
print(f'Die 수: train={len(xs_dict["train"]):,}, val={len(xs_dict["validation"]):,}, test={len(xs_dict["test"]):,}')

## 2. 실험 설정

In [ ]:
# ============================================================
# 실험 설정 — 모든 파라미터를 여기서 제어
# ============================================================

EXP_ID = '3-2-001'
EXP_TYPE = 'E2E Two-Stage HPO'
EXP_MEMO = 'End-to-End joint optimization (CLF+FS+REG)'

# --- Google Drive 파일 ID (실험 로그) ---
XLSX_GDRIVE_ID = '1IgaNh7ixgqpmH5PiwmSFbK2li6GODdew'
JSON_GDRIVE_ID = '1ycr6n5Ty_jzl4F-qQE4Cv5nS2WbIAZih'

# --- 전처리 ---
cleaning_params = dict(
    const_threshold=1e-6,
    missing_threshold=0.25,
    remove_duplicates=True,
    corr_threshold=0.95,
    add_indicator=True,
    indicator_threshold=0.01,
    imputation_method='median',
)

outlier_params = dict(
    method='winsorize',
    lower_pct=0.01,
    upper_pct=0.995,
)

# --- 샘플링 ---
sampling_params = dict(
    use_sampling=True,      # True: 빠른 탐색, False: 전체 데이터
    sample_frac=0.3,
)

# --- 웨이퍼맵 기반 피처 필터 ---
EXCLUDE_COLS = [
    # 예: 'X123', 'X456', ...
    # 웨이퍼맵 분석 결과를 여기에 추가
]

# ── 파이프라인 스위치 (pipeline_config) ──
# 이 dict의 모든 값이 JSON 실험 로그에 저장됩니다.
pipeline_config = dict(
    input_level='die',      # 'die' | 'unit' (unit이면 CLF 자동 OFF)
    run_clf=True,           # False → 분류 스킵, 회귀만
    clf_output='proba',     # 'proba' | 'binary' (P(Y>0) 확률 vs 0/1)
    clf_filter=False,       # True → 0 예측 샘플을 회귀에서 제외
    clf_optuna=True,        # False → CLF 기본 파라미터 (1회 캐싱)
    run_fs=True,            # False → Feature Selection 스킵
    fs_optuna=True,         # False → 고정 top_k (1회 캐싱)
    reg_level='unit',       # 'unit' | 'position'
    reg_optuna=True,        # False → REG 기본 파라미터 (1회 캐싱)
)

# ── E2E HPO 설정 ──
e2e_params = dict(
    clf_model='lgbm',
    reg_model='lgbm',
    n_trials=100,
    n_folds=3,
    clf_early_stop=50,
    reg_early_stop=50,
    imbalance_method='scale_pos_weight',
    agg_funcs=['mean', 'std', 'cv', 'range', 'min', 'max', 'median'],
    top_k_range=(50, 500),  # fs_optuna=True 일 때 탐색 범위
    top_k_fixed=200,        # fs_optuna=False 일 때 고정 값
    clf_fixed={},           # CLF 고정 파라미터 (탐색 제외)
    reg_fixed={},           # REG 고정 파라미터 (예: {'objective': 'poisson'})
)

# ── Rerun 설정 (best trial 재실행) ──
rerun_params = dict(
    n_folds=5,
    clf_early_stop=100,
    reg_early_stop=100,
)

LABEL_COL = 'label_bin'

# --- 설정 출력 ---
print(f'실험번호: {EXP_ID} | {EXP_TYPE}')
print(f'CLF: {e2e_params["clf_model"]} | REG: {e2e_params["reg_model"]}')
print(f'Trials: {e2e_params["n_trials"]} | Folds: {e2e_params["n_folds"]}')
print(f'Top-K: range={e2e_params["top_k_range"]}, fixed={e2e_params["top_k_fixed"]}')
print(f'샘플링: {"ON" if sampling_params["use_sampling"] else "OFF"} (frac={sampling_params["sample_frac"]})')
print(f'제외 컬럼: {len(EXCLUDE_COLS)}개')
print(f'\n── Pipeline Config ──')
for k, v in pipeline_config.items():
    print(f'  {k}: {v}')

download_from_drive(XLSX_GDRIVE_ID, JSON_GDRIVE_ID)
check_exp_id(EXP_ID)

## 3. 전처리 (클리닝 + 이상치)

In [ ]:
# --- Step 1: 클리닝 ---
xs_train, xs_val, xs_test, clean_cols, clean_report = run_cleaning(
    xs, feat_cols, xs_dict,
    **cleaning_params,
)

print(f'\n클리닝 결과: {len(feat_cols)} -> {len(clean_cols)}개')

# --- Step 2: 이상치 처리 ---
xs_train, xs_val, xs_test, outlier_report = run_outlier_treatment(
    xs_train, xs_val, xs_test, clean_cols,
    **outlier_params,
)

# --- Step 3: 웨이퍼맵 기반 피처 필터 ---
if EXCLUDE_COLS:
    before = len(clean_cols)
    clean_cols = [c for c in clean_cols if c not in set(EXCLUDE_COLS)]
    print(f'\n웨이퍼맵 필터: {before} -> {len(clean_cols)}개 ({before - len(clean_cols)}개 제외)')

print(f'\n최종 피처 수: {len(clean_cols)}개')

## 4. Health Merge + 라벨 생성 + Position 분리

In [ ]:
# --- Health merge ---
die_train = xs_train.merge(ys['train'], on=KEY_COL, how='left')
die_val = xs_val.merge(ys['validation'], on=KEY_COL, how='left')
die_test = xs_test.merge(ys['test'], on=KEY_COL, how='left')

assert die_train[TARGET_COL].notna().all(), 'train health NaN!'
assert die_val[TARGET_COL].notna().all(), 'val health NaN!'
assert die_test[TARGET_COL].notna().all(), 'test health NaN!'
print(f'Die-level merge: train={die_train.shape}, val={die_val.shape}, test={die_test.shape}')

# --- 샘플링 (unit 기준, train만) ---
USE_SAMPLING = sampling_params['use_sampling']
SAMPLE_FRAC = sampling_params['sample_frac']

if USE_SAMPLING:
    all_units = die_train[KEY_COL].drop_duplicates()
    sampled_units = all_units.sample(frac=SAMPLE_FRAC, random_state=SEED)
    n_before = len(die_train)
    die_train = die_train[die_train[KEY_COL].isin(sampled_units)].reset_index(drop=True)
    print(f'\n샘플링 ON (frac={SAMPLE_FRAC})')
    print(f'  Unit: {len(all_units):,} -> {len(sampled_units):,}')
    print(f'  Die: {n_before:,} -> {len(die_train):,}')
else:
    print('\n샘플링 OFF -> 전체 데이터 사용')

# --- 라벨 생성: 0 vs >0 ---
for df in [die_train, die_val, die_test]:
    df[LABEL_COL] = (df[TARGET_COL] > 0).astype(int)

print(f'\n라벨 분포 (train):')
dist = die_train[LABEL_COL].value_counts().sort_index()
for k, v in dist.items():
    print(f'  {k}: {v:,} ({v / len(die_train) * 100:.1f}%)')

# --- Position별 분리 ---
positions = sorted(die_train[POSITION_COL].unique())
feat_cols_clean = clean_cols

pos_data = {}
for pos in positions:
    pos_data[pos] = {
        'train': die_train[die_train[POSITION_COL] == pos].reset_index(drop=True),
        'val':   die_val[die_val[POSITION_COL] == pos].reset_index(drop=True),
        'test':  die_test[die_test[POSITION_COL] == pos].reset_index(drop=True),
    }
    n = {k: len(v) for k, v in pos_data[pos].items()}
    print(f'Position {pos}: train={n["train"]:,}, val={n["val"]:,}, test={n["test"]:,}')

print(f'\nClean feature 수: {len(feat_cols_clean)}개')

## 5. E2E Optuna HPO 실행

1 Trial = CLF → 집계 → FS(top-K) → REG → Val RMSE

In [ ]:
result = run_e2e_optimization(
    pos_data=pos_data,
    feat_cols=feat_cols_clean,
    pipeline_config=pipeline_config,
    label_col=LABEL_COL,
    **e2e_params,
)

study = result['study']
best_params = result['best_params']

print(f'\n── Best Trial ──')
print(f'Val RMSE: {result["best_value"]:.6f}')
print(f'\nBest params:')
for k, v in sorted(best_params.items()):
    print(f'  {k}: {v}')

## 6. Best Trial 재실행 (더 많은 Fold)

HPO에서 찾은 best params로 더 안정적인 예측을 생성합니다.

In [ ]:
final = rerun_best_trial(
    pos_data=pos_data,
    feat_cols=feat_cols_clean,
    best_params=best_params,
    pipeline_config=pipeline_config,
    clf_model=e2e_params['clf_model'],
    reg_model=e2e_params['reg_model'],
    label_col=LABEL_COL,
    imbalance_method=e2e_params['imbalance_method'],
    agg_funcs=e2e_params['agg_funcs'],
    top_k_fixed=e2e_params['top_k_fixed'],
    clf_fixed=e2e_params['clf_fixed'],
    reg_fixed=e2e_params['reg_fixed'],
    **rerun_params,
)

# --- Validation 평가 ---
y_val = final['unit_data']['val'][TARGET_COL].values
evaluate(y_val, final['val_pred'], label='E2E Rerun (val)')

## 7. 결과 시각화

In [ ]:
# --- Optuna 시각화 ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Trial RMSE 추이
trials = study.trials
vals = [t.value for t in trials if t.value is not None and t.value < float('inf')]
axes[0].plot(vals, 'o-', markersize=2, alpha=0.6)
axes[0].axhline(study.best_value, color='r', linestyle='--', label=f'Best: {study.best_value:.6f}')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel('Val RMSE')
axes[0].set_title('Trial 별 Val RMSE')
axes[0].legend()

# Feature importance (top 20)
if final.get('importances') is not None:
    imp_series = pd.Series(final['importances']).sort_values(ascending=True).tail(20)
    imp_series.plot(kind='barh', ax=axes[1])
    axes[1].set_title(f'Top 20 Features (of {len(final["selected_cols"])})')
    axes[1].set_xlabel('Importance (gain)')

plt.tight_layout()
plt.show()

## 8. 예측 CSV 생성

In [ ]:
from utils.config import OUTPUT_DIR

# --- Validation 예측 ---
val_df = final['unit_data']['val'][[KEY_COL]].copy()
val_df[TARGET_COL] = final['val_pred']
val_path = os.path.join(OUTPUT_DIR, 'e2e_val_pred.csv')
val_df.to_csv(val_path, index=False)
print(f'Val 예측 저장: {val_path} ({len(val_df):,}행)')

# --- Test 예측 ---
test_df = final['unit_data']['test'][[KEY_COL]].copy()
test_df[TARGET_COL] = final['test_pred']
test_path = os.path.join(OUTPUT_DIR, 'e2e_test_pred.csv')
test_df.to_csv(test_path, index=False)
print(f'Test 예측 저장: {test_path} ({len(test_df):,}행)')

print(f'\nVal RMSE: {final["val_rmse"]:.6f}')

## 9. 실험 로그

In [ ]:
# ============================================================
# 실험 로그 — 모든 파라미터 기록 (JSON + XLSX 증분 저장)
# ============================================================
log_experiment(
    exp_id=EXP_ID,
    exp_type=EXP_TYPE,
    best_model=f'{e2e_params["reg_model"]} (E2E)',
    val_rmse=final['val_rmse'],
    test_rmse=None,
    n_features=len(final['selected_cols']),
    memo=EXP_MEMO,
    cleaning_params=cleaning_params,
    outlier_params=outlier_params,
    feature_sel_params={
        'method': 'e2e_top_k',
        'top_k': best_params.get('top_k'),
        'top_k_fixed': e2e_params['top_k_fixed'],
        'top_k_range': e2e_params['top_k_range'],
    },
    agg_params={
        'agg_funcs': e2e_params['agg_funcs'],
    },
    model_params={
        'pipeline': 'E2E Joint HPO',
        'pipeline_config': pipeline_config,
        'clf_model': e2e_params['clf_model'],
        'reg_model': e2e_params['reg_model'],
        'n_trials': e2e_params['n_trials'],
        'n_folds_hpo': e2e_params['n_folds'],
        'n_folds_rerun': rerun_params['n_folds'],
        'imbalance': e2e_params['imbalance_method'],
        'sampling': sampling_params,
        'exclude_cols': EXCLUDE_COLS,
        'best_params': best_params,
        'best_val_rmse_hpo': result['best_value'],
        'best_val_rmse_rerun': final['val_rmse'],
        'clf_fixed': e2e_params['clf_fixed'],
        'reg_fixed': e2e_params['reg_fixed'],
    },
    feature_cols=final['selected_cols'],
    xlsx_gdrive_id=XLSX_GDRIVE_ID,
    json_gdrive_id=JSON_GDRIVE_ID,
)